# 01 Data Inventory

Notebook-first article reproduction. Calculations are kept in notebook cells.

## Inventory
Validate the local input table and write machine-readable provenance tables.

In [1]:

from pathlib import Path
import json
import math
import pickle
import shutil
import subprocess
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"
RESULTS = ROOT / "results"
TABLES = RESULTS / "tables"
FIGURES = RESULTS / "figures"
MANUSCRIPT_FIGURES = FIGURES / "manuscript"
MODELS_DIR = RESULTS / "models"
SPLITS_DIR = RESULTS / "splits"
MANUSCRIPT = ROOT / "manuscript"
GENERATED = MANUSCRIPT / "tables"

for path in [TABLES, MANUSCRIPT_FIGURES, MODELS_DIR, SPLITS_DIR, GENERATED]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_COL = "-lgLD50, mol/kg"
CLUSTER_COL = "Butina_clusters"
BBB_FILTER_COL = "bbb_rule_pass"
LIGAND_ID_COL = "ligand_id"
FP_PREFIX = "FP_"
RANDOM_STATE = 42

PROTEINS = [
    "1m2z", "1pbq", "1xoq", "2rh1", "2vt4", "2ydo", "2z5x", "3b66", "3kk6", "3ln1",
    "3rze", "4djh", "4ey7", "4iar", "4mqs", "4n6h", "5cxv", "5i71", "5tvn", "5u09",
    "5va1", "6cm4", "6kpf", "6kux", "6lqa", "6pdj", "6x3x", "6y1z", "7f8y", "7kwe",
    "7ljd", "7wc9", "7xnk", "7ym8", "8e9y", "8ef6", "8fhs", "8pjk", "8st0", "8wty",
    "8xvk", "8yn3", "9eo4", "V1A",
]
MODEL_ORDER = ["Baseline", "PCA", "ADME", "Plain", "BBB pass"]


In [2]:

df_path = DATA / "df_final.csv"
if not df_path.exists():
    raise FileNotFoundError(df_path)

df = pd.read_csv(df_path)
print("df_final.csv shape:", df.shape)
if tuple(df.shape) != (12651, 2237):
    raise AssertionError(f"Unexpected df_final.csv shape: {df.shape}")

required = {TARGET_COL, BBB_FILTER_COL, LIGAND_ID_COL, "MW", "MW, g/mol", "logP", *PROTEINS}
missing = sorted(required.difference(df.columns))
if missing:
    raise AssertionError(f"Missing required columns: {missing}")

exclusion_registry = pd.read_csv(DATA / "excluded_molecules.csv")
excluded_model_ids = sorted(exclusion_registry.loc[exclusion_registry["scope"].eq("all_model_splits"), LIGAND_ID_COL].astype(int).tolist())
modeling_df = df.loc[~df[LIGAND_ID_COL].astype(int).isin(excluded_model_ids)].copy()
if len(modeling_df) != 12650 or set(excluded_model_ids) & set(modeling_df[LIGAND_ID_COL].astype(int)):
    raise AssertionError("The active modeling cohort must exclude every all_model_splits ligand")

fp_cols = sorted([c for c in df.columns if c.startswith(FP_PREFIX)], key=lambda x: int(x.split("_")[1]))
if len(fp_cols) != 2048:
    raise AssertionError(f"Expected 2048 fingerprint columns, found {len(fp_cols)}")

protein_nan_total = int(df[PROTEINS].isna().sum().sum())
protein_rows_any_nan = int(df[PROTEINS].isna().any(axis=1).sum())
print("protein NaN cells:", protein_nan_total)
print("rows with any protein NaN:", protein_rows_any_nan)

inventory = pd.DataFrame([
    {"item": "raw_source_rows", "value": int(df.shape[0])},
    {"item": "excluded_all_model_splits", "value": len(excluded_model_ids)},
    {"item": "active_modeling_cohort", "value": int(modeling_df.shape[0])},
    {"item": "n_columns", "value": int(df.shape[1])},
    {"item": "n_fingerprint_columns", "value": int(len(fp_cols))},
    {"item": "n_protein_columns", "value": int(len(PROTEINS))},
    {"item": "protein_nan_cells", "value": protein_nan_total},
    {"item": "rows_with_any_protein_nan", "value": protein_rows_any_nan},
    {"item": "target_nonmissing", "value": int(df[TARGET_COL].notna().sum())},
    {"item": "raw_unique_ligand_id", "value": int(df[LIGAND_ID_COL].nunique())},
    {"item": "active_unique_ligand_id", "value": int(modeling_df[LIGAND_ID_COL].nunique())},
])
inventory.to_csv(TABLES / "data_inventory.csv", index=False)

protein_nan = pd.DataFrame({
    "protein": PROTEINS,
    "n_nan": [int(df[c].isna().sum()) for c in PROTEINS],
    "n_nonmissing": [int(df[c].notna().sum()) for c in PROTEINS],
})
protein_nan.to_csv(TABLES / "protein_nan_summary.csv", index=False)

feature_inventory = pd.DataFrame([
    {"feature_group": "fingerprint", "n_columns": len(fp_cols), "columns": ";".join(fp_cols)},
    {"feature_group": "protein", "n_columns": len(PROTEINS), "columns": ";".join(PROTEINS)},
    {"feature_group": "plain_physchem", "n_columns": 2, "columns": "MW, g/mol;logP"},
    {"feature_group": "conformal_plain_physchem", "n_columns": 2, "columns": "MW;logP"},
])
feature_inventory.to_csv(TABLES / "feature_inventory.csv", index=False)
print(inventory.to_string(index=False))


df_final.csv shape: (12651, 2237)


protein NaN cells: 8159
rows with any protein NaN: 3354
                     item  value
          raw_source_rows  12651
excluded_all_model_splits      1
   active_modeling_cohort  12650
                n_columns   2237
    n_fingerprint_columns   2048
        n_protein_columns     44
        protein_nan_cells   8159
rows_with_any_protein_nan   3354
        target_nonmissing  12651
     raw_unique_ligand_id  12651
  active_unique_ligand_id  12650


## Rebuild Butina clusters and the split registry

This cell is the canonical regeneration path for the structural clusters and the 30 cluster-safe train/validation/test assignments. It reads the unmodified source table and exclusion registry, then verifies the generated cluster and split-membership checksums against the frozen protocol. No model fitting or test-label-based choice occurs in this step.


In [3]:
import hashlib
import platform
import sys

from rdkit import Chem, DataStructs, rdBase
from rdkit.Chem import rdFingerprintGenerator
from rdkit.ML.Cluster import Butina
from sklearn import __version__ as sklearn_version
from sklearn.model_selection import GroupShuffleSplit

BUTINA_DIR = SPLITS_DIR / "butina_r2_fp2048_achiral_d035_excluded1298"
EXPECTED_CLUSTERS_SHA256 = "c75e1d3ad90505553a580aeb3ce95fd86ff425350948e42a5f87858ddd5d61f1"
EXPECTED_CLUSTER_SUMMARY_SHA256 = "dbf6f9b90dff4ae94e5830ad6bcf072993c87298964c9354a543d6ff55bea44a"
EXPECTED_REGISTRY_SHA256 = "54b048fe612bddecfc863af2565c8070cc7665de33dad179a93263ecf2706715"


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def split_signature(split_frame: pd.DataFrame) -> str:
    payload = split_frame[["set", "ligand_id"]].to_csv(index=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()


exclusions = pd.read_csv(DATA / "excluded_molecules.csv")
excluded_ids = sorted(exclusions.loc[exclusions["scope"].eq("all_model_splits"), LIGAND_ID_COL].astype(int).tolist())
cluster_input = df[[LIGAND_ID_COL, "Canonical SMILES"]].copy()
cluster_input[LIGAND_ID_COL] = cluster_input[LIGAND_ID_COL].astype(int)
cluster_input = cluster_input.loc[~cluster_input[LIGAND_ID_COL].isin(excluded_ids)].sort_values(LIGAND_ID_COL, kind="stable").reset_index(drop=True)
if cluster_input[LIGAND_ID_COL].duplicated().any() or cluster_input["Canonical SMILES"].isna().any():
    raise ValueError("Cluster input must have unique ligand IDs and non-missing canonical SMILES")

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(
    radius=2, fpSize=2048, includeChirality=False, countSimulation=False
)
fingerprints = []
for row in cluster_input.itertuples(index=False):
    molecule = Chem.MolFromSmiles(getattr(row, "_1"))
    if molecule is None:
        raise ValueError(f"Cannot parse canonical SMILES for ligand_id={getattr(row, LIGAND_ID_COL)}")
    fingerprints.append(fingerprint_generator.GetFingerprint(molecule))

n_molecules = len(fingerprints)
n_distances = n_molecules * (n_molecules - 1) // 2
distances = np.empty(n_distances, dtype=np.float64)
cursor = 0
for index in range(1, n_molecules):
    similarities = np.asarray(DataStructs.BulkTanimotoSimilarity(fingerprints[index], fingerprints[:index]), dtype=np.float64)
    next_cursor = cursor + len(similarities)
    distances[cursor:next_cursor] = 1.0 - similarities
    cursor = next_cursor
    if index % 2000 == 0:
        print(f"Calculated distances for {index:,}/{n_molecules:,} molecules.")
if cursor != n_distances:
    raise AssertionError(f"Expected {n_distances:,} distances, wrote {cursor:,}")

raw_clusters = Butina.ClusterData(distances, n_molecules, 0.35, isDistData=True, reordering=False)
cluster_ids = np.empty(n_molecules, dtype=np.int32)
cluster_sizes = np.empty(n_molecules, dtype=np.int32)
cluster_centers = np.empty(n_molecules, dtype=np.int64)
cluster_summary_rows = []
for cluster_id, members in enumerate(raw_clusters, start=1):
    member_indices = np.asarray(members, dtype=int)
    # The cluster center is defined as the stable minimum ligand ID in each RDKit cluster.
    center_ligand_id = int(cluster_input.iloc[member_indices][LIGAND_ID_COL].min())
    cluster_ids[member_indices] = cluster_id
    cluster_sizes[member_indices] = len(member_indices)
    cluster_centers[member_indices] = center_ligand_id
    cluster_summary_rows.append({
        CLUSTER_COL: cluster_id,
        "cluster_size": len(member_indices),
        "cluster_center_ligand_id": center_ligand_id,
    })

clusters = cluster_input.copy()
clusters.insert(0, "input_order", np.arange(n_molecules, dtype=int))
clusters[CLUSTER_COL] = cluster_ids
clusters["cluster_size"] = cluster_sizes
clusters["cluster_center_ligand_id"] = cluster_centers
cluster_summary = pd.DataFrame(cluster_summary_rows)
if clusters[CLUSTER_COL].nunique() != len(cluster_summary) or len(clusters) != 12650:
    raise AssertionError("Unexpected Butina cluster count or molecule count")

BUTINA_DIR.mkdir(parents=True, exist_ok=True)
clusters_path = BUTINA_DIR / "clusters.csv"
cluster_summary_path = BUTINA_DIR / "cluster_summary.csv"
clusters.to_csv(clusters_path, index=False)
cluster_summary.to_csv(cluster_summary_path, index=False)
clusters_sha256 = sha256_file(clusters_path)
cluster_summary_sha256 = sha256_file(cluster_summary_path)
if clusters_sha256 != EXPECTED_CLUSTERS_SHA256 or cluster_summary_sha256 != EXPECTED_CLUSTER_SUMMARY_SHA256:
    raise AssertionError("Butina regeneration differs from the frozen protocol artifacts")

cluster_validation = {
    "success": True,
    "df_final_rows_after_rewrite": int(len(df)),
    "excluded_ids": excluded_ids,
    "excluded_ids_not_assigned": not set(excluded_ids) & set(clusters[LIGAND_ID_COL].astype(int)),
    "assigned_non_excluded_molecules": int(len(clusters)),
    "n_clustered_molecules": int(len(clusters)),
    "n_clusters": int(clusters[CLUSTER_COL].nunique()),
    "n_singleton_clusters": int((cluster_summary["cluster_size"] == 1).sum()),
    "largest_cluster_size": int(cluster_summary["cluster_size"].max()),
    "clusters_sha256": clusters_sha256,
    "cluster_summary_sha256": cluster_summary_sha256,
}
with (BUTINA_DIR / "validation.json").open("w") as handle:
    json.dump(cluster_validation, handle, indent=2)

cluster_metadata = {
    "protocol_name": "butina_r2_fp2048_achiral_d035_excluded1298",
    "purpose": "Reviewer-facing Butina clustering after applying excluded_molecules.csv.",
    "input": {
        "df_final": "data/df_final.csv",
        "df_final_sha256_after_rewrite": sha256_file(DATA / "df_final.csv"),
        "excluded_molecules": "data/excluded_molecules.csv",
        "excluded_molecules_sha256": sha256_file(DATA / "excluded_molecules.csv"),
        "excluded_ligand_ids": excluded_ids,
        "molecules_before_exclusion": int(len(df)),
        "molecules_clustered": int(len(clusters)),
        "smiles_column": "Canonical SMILES",
        "input_order": "ascending ligand_id after applying exclusions",
    },
    "fingerprint": {"type": "Morgan bit vector", "radius": 2, "fpSize": 2048, "includeChirality": False, "countSimulation": False},
    "distance": {"metric": "1 - TanimotoSimilarity", "dtype": "float64", "similarity_threshold": 0.65, "distance_cutoff": 0.35},
    "butina": {"rdkit_call": "Butina.ClusterData(dist, n, 0.35, isDistData=True, reordering=False)", "isDistData": True, "reordering": False, "cluster_numbering": "RDKit output order, 1-based"},
    "environment": {"python": sys.version, "platform": platform.platform(), "rdkit": rdBase.rdkitVersion, "numpy": np.__version__, "pandas": pd.__version__},
    "outputs": {"directory": "results/splits/butina_r2_fp2048_achiral_d035_excluded1298", "clusters": "clusters.csv", "cluster_summary": "cluster_summary.csv", "validation": "validation.json"},
    "result": cluster_validation,
}
with (BUTINA_DIR / "metadata.json").open("w") as handle:
    json.dump(cluster_metadata, handle, indent=2)

split_rows = []
split_summary_rows = []
all_indices = np.arange(len(clusters))
group_values = clusters[CLUSTER_COL].to_numpy()
for split_index, seed in enumerate(range(42, 72)):
    first_stage = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    development_indices, test_indices = next(first_stage.split(all_indices, groups=group_values))
    second_stage = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    train_local_indices, val_local_indices = next(second_stage.split(development_indices, groups=group_values[development_indices]))
    set_indices = {
        "train": development_indices[train_local_indices],
        "val": development_indices[val_local_indices],
        "test": test_indices,
    }
    cluster_sets = {name: set(group_values[indices]) for name, indices in set_indices.items()}
    shared_train_val = len(cluster_sets["train"] & cluster_sets["val"])
    shared_train_test = len(cluster_sets["train"] & cluster_sets["test"])
    shared_val_test = len(cluster_sets["val"] & cluster_sets["test"])
    if shared_train_val or shared_train_test or shared_val_test:
        raise AssertionError(f"Split {split_index} shares one or more Butina clusters")
    split_frame = pd.concat([
        pd.DataFrame({"set": name, LIGAND_ID_COL: np.sort(clusters.iloc[indices][LIGAND_ID_COL].to_numpy(dtype=int))})
        for name, indices in set_indices.items()
    ], ignore_index=True)
    if split_frame[LIGAND_ID_COL].nunique() != len(clusters) or len(split_frame) != len(clusters):
        raise AssertionError(f"Split {split_index} does not assign every included molecule exactly once")
    split_frame.insert(0, "split_index", split_index)
    split_rows.append(split_frame)
    split_summary_rows.append({
        "split_index": split_index,
        "seed": seed,
        "n_molecules": int(len(clusters)),
        "n_clusters": int(clusters[CLUSTER_COL].nunique()),
        "split_signature_sha256": split_signature(split_frame),
        "shared_clusters_train_val": shared_train_val,
        "shared_clusters_train_test": shared_train_test,
        "shared_clusters_val_test": shared_val_test,
        "n_train": int(len(set_indices["train"])),
        "n_clusters_train": int(len(cluster_sets["train"])),
        "n_val": int(len(set_indices["val"])),
        "n_clusters_val": int(len(cluster_sets["val"])),
        "n_test": int(len(set_indices["test"])),
        "n_clusters_test": int(len(cluster_sets["test"])),
    })

split_registry = pd.concat(split_rows, ignore_index=True)
split_summary = pd.DataFrame(split_summary_rows)
split_registry_path = SPLITS_DIR / "split_indices.csv"
split_summary_path = SPLITS_DIR / "split_summary.csv"
split_registry.to_csv(split_registry_path, index=False)
split_summary.to_csv(split_summary_path, index=False)
registry_sha256 = sha256_file(split_registry_path)
if registry_sha256 != EXPECTED_REGISTRY_SHA256:
    raise AssertionError("Cluster-safe registry differs from the frozen protocol artifact")

split_validation = {
    "success": True,
    "n_splits": int(len(split_summary)),
    "seeds": split_summary["seed"].astype(int).tolist(),
    "molecules_per_split": int(len(clusters)),
    "clusters_per_split": int(clusters[CLUSTER_COL].nunique()),
    "all_split_signatures_distinct": bool(split_summary["split_signature_sha256"].is_unique),
    "registry_rows": int(len(split_registry)),
    "registry_sha256": registry_sha256,
    "summary_sha256": sha256_file(split_summary_path),
    "max_shared_clusters_between_sets": int(split_summary[["shared_clusters_train_val", "shared_clusters_train_test", "shared_clusters_val_test"]].to_numpy().max()),
}
with (SPLITS_DIR / "split_validation.json").open("w") as handle:
    json.dump(split_validation, handle, indent=2)

split_metadata = {
    "protocol_name": "butina_cluster_safe_random_splits_v1",
    "purpose": "Thirty reproducible cluster-safe random splits for downstream model work.",
    "input": {"clusters": "results/splits/butina_r2_fp2048_achiral_d035_excluded1298/clusters.csv", "clusters_sha256": clusters_sha256, "input_order": "ascending ligand_id from clusters.csv", "n_molecules": int(len(clusters)), "n_clusters": int(clusters[CLUSTER_COL].nunique())},
    "algorithm": {"class": "sklearn.model_selection.GroupShuffleSplit", "first_stage": {"groups": CLUSTER_COL, "test_size": 0.2, "random_state": "seed for each split", "output": "development and held-out test"}, "second_stage": {"input": "development", "groups": CLUSTER_COL, "test_size": 0.2, "random_state": "same seed for each split", "output": "train and validation"}, "set_order": ["train", "val", "test"], "split_indices": list(range(30)), "seeds": list(range(42, 72))},
    "environment": {"python": sys.version, "platform": platform.platform(), "numpy": np.__version__, "pandas": pd.__version__, "scikit_learn": sklearn_version},
    "outputs": {"registry": "split_indices.csv", "summary": "split_summary.csv", "validation": "split_validation.json"},
    "result": split_validation,
}
with (SPLITS_DIR / "split_metadata.json").open("w") as handle:
    json.dump(split_metadata, handle, indent=2)

print(f"Rebuilt {len(cluster_summary):,} Butina clusters and {len(split_summary)} cluster-safe splits.")
print(f"Cluster checksum: {clusters_sha256}")
print(f"Registry checksum: {registry_sha256}")


Calculated distances for 2,000/12,650 molecules.


Calculated distances for 4,000/12,650 molecules.


Calculated distances for 6,000/12,650 molecules.


Calculated distances for 8,000/12,650 molecules.


Calculated distances for 10,000/12,650 molecules.


Calculated distances for 12,000/12,650 molecules.


Rebuilt 8,195 Butina clusters and 30 cluster-safe splits.
Cluster checksum: c75e1d3ad90505553a580aeb3ce95fd86ff425350948e42a5f87858ddd5d61f1
Registry checksum: 54b048fe612bddecfc863af2565c8070cc7665de33dad179a93263ecf2706715


In [4]:

support_rows = []
support_files = [
    ("data/excluded_molecules.csv", DATA / "excluded_molecules.csv"),
    ("data/receptor_panel_annotation.csv", DATA / "receptor_panel_annotation.csv"),
    ("results/splits/split_indices.csv", SPLITS_DIR / "split_indices.csv"),
]
for name, path in support_files:
    support_rows.append({"file": name, "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else 0})
    if not path.exists():
        raise FileNotFoundError(path)

pd.DataFrame(support_rows).to_csv(TABLES / "support_files_inventory.csv", index=False)
print(pd.DataFrame(support_rows).to_string(index=False))


                              file  exists   bytes
       data/excluded_molecules.csv    True     254
data/receptor_panel_annotation.csv    True    2881
  results/splits/split_indices.csv    True 5099538


## Structural separation audit

This audit asks whether a molecule reserved for validation or final testing has a close structural analogue in the model-training data. For each query molecule, we calculate its largest Tanimoto similarity to the relevant reference set using the same Morgan fingerprint representation used for Butina clustering. A value of 0 means no shared fingerprint bits and 1 means identical fingerprints.

The result for the 30 predefined splits is reported as the mean plus or minus the sample standard deviation of a statistic calculated separately in each split. The pooled molecule-level value is retained only as a technical cross-check; it is not the manuscript result.

In [5]:
from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator

SPLIT_REGISTRY = SPLITS_DIR / "split_indices.csv"
SPLIT_COMPARISONS = (("val", "train"), ("test", "train"), ("test", "val"))
SIMILARITY_THRESHOLDS = (0.70, 0.80, 0.90)
SIMILARITY_STATISTICS = ("median", "p90", "p95", "p99", "maximum", "fraction_ge_0.70", "fraction_ge_0.80", "fraction_ge_0.90")

split_manifest = pd.read_csv(SPLIT_REGISTRY)
split_manifest["seed"] = split_manifest["split_index"].astype(int) + 42
structures = pd.read_csv(DATA / "df_final.csv", usecols=["ligand_id", "Canonical SMILES"]).rename(columns={"Canonical SMILES": "canonical_smiles"})
split_manifest = split_manifest.merge(structures, on="ligand_id", how="left", validate="many_to_one")
required = {"split_index", "seed", "set", "ligand_id", "canonical_smiles"}
missing = required - set(split_manifest.columns)
if missing:
    raise ValueError(f"Split registry lacks columns: {sorted(missing)}")

split_structures = split_manifest[["ligand_id", "canonical_smiles"]].drop_duplicates()
if len(split_structures) != 12650 or not split_structures["ligand_id"].is_unique:
    raise ValueError("Expected 12,650 unique structures in the split registry")

fingerprint_generator = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048, includeChirality=False)
fingerprints = {}
for row in split_structures.itertuples(index=False):
    molecule = Chem.MolFromSmiles(row.canonical_smiles)
    if molecule is None:
        raise ValueError(f"Cannot parse canonical SMILES for ligand_id={row.ligand_id}")
    fingerprints[int(row.ligand_id)] = fingerprint_generator.GetFingerprint(molecule)

print(f"Prepared Morgan fingerprints for {len(fingerprints):,} molecules.")


Prepared Morgan fingerprints for 12,650 molecules.


In [6]:
similarity_rows = []
for split_index, split in split_manifest.groupby("split_index", sort=True):
    seed = int(split["seed"].iloc[0])
    ids_by_set = {name: group["ligand_id"].astype(int).tolist() for name, group in split.groupby("set")}
    for query_set, reference_set in SPLIT_COMPARISONS:
        reference_ids = ids_by_set[reference_set]
        reference_fps = [fingerprints[ligand_id] for ligand_id in reference_ids]
        for ligand_id in ids_by_set[query_set]:
            similarities = DataStructs.BulkTanimotoSimilarity(fingerprints[ligand_id], reference_fps)
            max_similarity = max(similarities)
            nearest_ligand_id = min(reference_ids[i] for i, value in enumerate(similarities) if value == max_similarity)
            similarity_rows.append({
                "split_index": int(split_index),
                "seed": seed,
                "query_set": query_set,
                "reference_set": reference_set,
                "ligand_id": ligand_id,
                "nearest_ligand_id": nearest_ligand_id,
                "max_tanimoto_similarity": max_similarity,
            })

nearest_neighbors = pd.DataFrame(similarity_rows)
nearest_neighbors.to_csv(TABLES / "nearest_neighbor_similarity.csv", index=False)
print(f"Saved {len(nearest_neighbors):,} query/reference comparisons.")


Saved 212,463 query/reference comparisons.


In [7]:
def similarity_summary(values):
    summary = {
        "n_queries": int(len(values)),
        "median": float(values.median()),
        "p90": float(values.quantile(0.90)),
        "p95": float(values.quantile(0.95)),
        "p99": float(values.quantile(0.99)),
        "maximum": float(values.max()),
    }
    for threshold in SIMILARITY_THRESHOLDS:
        summary[f"fraction_ge_{threshold:.2f}"] = float((values >= threshold).mean())
    return summary

summary_rows = []
for keys, group in nearest_neighbors.groupby(["split_index", "seed", "query_set", "reference_set"], sort=True):
    split_index, seed, query_set, reference_set = keys
    summary_rows.append({
        "scope": "per_split", "split_index": split_index, "seed": seed,
        "query_set": query_set, "reference_set": reference_set,
        **similarity_summary(group["max_tanimoto_similarity"]),
    })
for keys, group in nearest_neighbors.groupby(["query_set", "reference_set"], sort=True):
    query_set, reference_set = keys
    summary_rows.append({
        "scope": "all_molecules_pooled", "split_index": "", "seed": "",
        "query_set": query_set, "reference_set": reference_set,
        **similarity_summary(group["max_tanimoto_similarity"]),
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(TABLES / "nearest_neighbor_summary.csv", index=False)

across_split_rows = []
for keys, group in summary.query("scope == 'per_split'").groupby(["query_set", "reference_set"], sort=True):
    query_set, reference_set = keys
    for statistic in SIMILARITY_STATISTICS:
        values = group[statistic]
        across_split_rows.append({
            "query_set": query_set, "reference_set": reference_set, "statistic": statistic,
            "n_splits": int(len(values)),
            "mean_across_splits": float(values.mean()),
            "std_across_splits": float(values.std(ddof=1)),
            "minimum_across_splits": float(values.min()),
            "maximum_across_splits": float(values.max()),
        })

across_splits = pd.DataFrame(across_split_rows)
across_splits.to_csv(TABLES / "nearest_neighbor_summary_across_splits.csv", index=False)

summary_display = across_splits.pivot(index=["query_set", "reference_set"], columns="statistic", values=["mean_across_splits", "std_across_splits"])
print("Results below are mean +/- sample SD across 30 splits; fractions are proportions.")
display(summary_display)


Results below are mean +/- sample SD across 30 splits; fractions are proportions.


mean_across_splits                                    \
statistic                 fraction_ge_0.70 fraction_ge_0.80 fraction_ge_0.90   
query_set reference_set                                                        
test      train                   0.075541         0.025173         0.002507   
          val                     0.020084         0.005923         0.000579   
val       train                   0.078702         0.024631         0.002586   

                                                                           \
statistic                 maximum    median       p90       p95       p99   
query_set reference_set                                                     
test      train          0.964335  0.536963  0.665874  0.739687  0.846204   
          val            0.926742  0.416206  0.595018  0.633733  0.755941   
val       train          0.965034  0.538286  0.671827  0.742692  0.845894   

                        std_across_splits                                    \
statistic                fraction_ge_0.70 fraction_ge_0.80 fraction_ge_0.90   
query_set reference_set                                                       
test      train                  0.006367         0.003698         0.001440   
          val                    0.004704         0.001642         0.000741   
val       train                  0.010085         0.004844         0.001797   

                                                                           
statistic                 maximum    median       p90       p95       p99  
query_set reference_set                                                    
test      train          0.019543  0.003322  0.010025  0.008472  0.009617  
          val            0.044612  0.004769  0.007065  0.006187  0.021197  
val       train          0.021531  0.005126  0.013940  0.012390  0.011697

In [8]:
def summary_value(query_set, reference_set, statistic, percent=False):
    row = across_splits.loc[(across_splits["query_set"] == query_set) & (across_splits["reference_set"] == reference_set) & (across_splits["statistic"] == statistic)].iloc[0]
    scale = 100 if percent else 1
    suffix = r"\%" if percent else ""
    return f"{row['mean_across_splits'] * scale:.3f} $\pm$ {row['std_across_splits'] * scale:.3f}{suffix}"

latex_rows = []
for query_set, reference_set, label in [("val", "train", r"Validation $\rightarrow$ train"), ("test", "train", r"Test $\rightarrow$ train"), ("test", "val", r"Test $\rightarrow$ validation")]:
    latex_rows.append(" & ".join([label, summary_value(query_set, reference_set, "median"), summary_value(query_set, reference_set, "p90"), summary_value(query_set, reference_set, "p95"), summary_value(query_set, reference_set, "p99"), summary_value(query_set, reference_set, "maximum"), summary_value(query_set, reference_set, "fraction_ge_0.80", percent=True), summary_value(query_set, reference_set, "fraction_ge_0.90", percent=True)]) + r" \\")

bs = chr(92)
table_s4 = "\n".join([
    "% Auto-generated by notebooks/01_data_inventory.ipynb.",
    f"{bs}begin{{table}}[!htbp]",
    f"{bs}centering",
    f"{bs}scriptsize",
    f"{bs}setlength{{{bs}tabcolsep}}{{3pt}}",
    f"{bs}caption{{Residual structural similarity across the 30 Butina-cluster-safe splits. For each query molecule, the maximum Morgan fingerprint Tanimoto similarity to the indicated reference set was calculated. Every cell is the mean ${bs}pm$ sample standard deviation of the split-level statistic across 30 splits; it is not a pooled molecule-level estimate.}}{bs}label{{tab:S4_split_similarity}}",
    f"{bs}begin{{tabular}}{{lrrrrrrr}}",
    f"{bs}toprule",
    f"Comparison & Median & P90 & P95 & P99 & Maximum & ${bs}geq0.80$ & ${bs}geq0.90$ {bs}{bs}",
    f"{bs}midrule",
    *latex_rows,
    f"{bs}bottomrule",
    f"{bs}end{{tabular}}",
    f"{bs}end{{table}}",
]) + "\n"
snippet = rf"""% Auto-generated by notebooks/01_data_inventory.ipynb.
We used 30 predefined Butina-cluster-disjoint train/validation/test splits (Morgan radius 2, 2,048 bits, achiral fingerprints; Tanimoto-distance cutoff 0.35). To quantify residual structural proximity not captured by cluster membership, we identified the nearest reference analogue for every validation and test molecule using maximum Tanimoto similarity (Supplementary Table S4). Across the 30 splits, the test-to-train median nearest-neighbour similarity was {summary_value('test', 'train', 'median')}; the P95 was {summary_value('test', 'train', 'p95')}. The fraction of test molecules with a nearest training analogue of at least 0.80 was {summary_value('test', 'train', 'fraction_ge_0.80', percent=True)}, and that at least 0.90 was {summary_value('test', 'train', 'fraction_ge_0.90', percent=True)}. Thus, cluster-disjoint splitting removed shared cluster membership but did not create a strictly out-of-domain test set; residual close analogues are reported as a limitation rather than removed by post hoc split tuning.
"""
(GENERATED / "tableS4_cross_split_similarity.tex").write_text(table_s4)
(GENERATED / "split_independence_text.tex").write_text(snippet)
print("Wrote manuscript/tables/tableS4_cross_split_similarity.tex and split_independence_text.tex")


Wrote manuscript/tables/tableS4_cross_split_similarity.tex and split_independence_text.tex
